<a href="https://colab.research.google.com/github/fedetorres91/zero-to-hero-nn-notebooks-play/blob/main/calculator_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
# build the vocabulary of digits and signs and mappings to/from integers
chars = list(str(idx) for idx in range(10))
chars.append("+")
chars.append("=")
chars.append(";") #semicolon separates equations
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s,i in stoi.items()}
# create a mapping from characters to integers
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers

def encode_equation(equation):

    for idx, ch in enumerate(equation):
        if ch == "=":
            start_index = idx + 1
    end_index = len(equation) - 1

    # reverse solution digits and attach final token
    # get solution number

    #get units, tens, hundretdhs
    # decimals = get_decimal_digits(int(equation[start_index:-1]))
    return [stoi[idx] for idx in equation[:start_index]] + [stoi[idx] for idx in equation[start_index:-1]][::-1] + [stoi[equation[-1]]]

decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
def decode_equation(l):

    for idx, tk in enumerate(l):
        if tk == 11:
            start_index = idx + 1
    end_index = len(l) - 1

    # reverse solution digits and attach final token
    # get solution number

    #get units, tens, hundretdhs
    # decimals = get_decimal_digits(int(equation[start_index:-1]))
    return "".join([itos[idx] for idx in l[:start_index]] + [itos[idx] for idx in l[start_index:-1]][::-1] + [itos[l[-1]]])
vocab_size = len(itos)

In [ ]:
# encode solution reversing order
def get_decimal_digits(number): # number < 10000
  decimals = {}
  number = int(number)
  while number != 0:
    decimals[1] = str(number % 10)
    number = number // 10
    decimals[10] = str(number % 10)
    number = number // 10
    decimals[100] = str(number % 10)
    number = number // 10
    decimals[1000] = str(number % 10)
    number = number //10
  return decimals

In [ ]:
vocab_sie=len(itos)
vocab_size

13

In [ ]:
# create equations of integers between min and max, and create labels x = [a, b], y = [a+b]
def create_equations(size, min, max, with_solutions=True):
  equations = []
  for i in range(size):
    a = random.randint(min, max)
    b = random.randint(min, max)
    if with_solutions:
      equations.append(f";;;{a:02}+{b:02}={a + b:03};") # (;nn+nn=nnn;;) attach ;;; as padding to have fixed context size(12)
    else:
      equations.append(f";;;{a:02}+{b:02}=") # (;nn+nn=) attach ; as padding to have fixed context size(9) for generating equations without solutions
  return equations

In [ ]:
equations = create_equations(1, 0, 99, with_solutions=False)
encode(equations[0])

[12, 12, 12, 1, 6, 10, 8, 5, 11]

In [ ]:
# create data
random.seed(42); # seed rng for reproducibility
equations = create_equations(1000, 0, 99)
encoded_equations = [encode_equation(equation) for equation in equations]
#final_equations = [item for encoded_equation in encoded_equations for item in encoded_equation]
data = torch.tensor(encoded_equations, dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
encoded_equations[:10]

[[12, 12, 12, 8, 1, 10, 1, 4, 11, 5, 9, 0, 12],
 [12, 12, 12, 0, 3, 10, 9, 4, 11, 7, 9, 0, 12],
 [12, 12, 12, 3, 5, 10, 3, 1, 11, 6, 6, 0, 12],
 [12, 12, 12, 2, 8, 10, 1, 7, 11, 5, 4, 0, 12],
 [12, 12, 12, 9, 4, 10, 1, 3, 11, 7, 0, 1, 12],
 [12, 12, 12, 8, 6, 10, 9, 4, 11, 0, 8, 1, 12],
 [12, 12, 12, 6, 9, 10, 1, 1, 11, 0, 8, 0, 12],
 [12, 12, 12, 7, 5, 10, 5, 4, 11, 9, 2, 1, 12],
 [12, 12, 12, 0, 4, 10, 0, 3, 11, 7, 0, 0, 12],
 [12, 12, 12, 1, 1, 10, 2, 7, 11, 8, 3, 0, 12]]

In [ ]:
batch_size = 32
batch_size = batch_size//4 # correct for each batch(equation) we generate 4 batches.
block_size = 9 # (;;;nn+nn=)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # generate a small batch of data of inputs x and targets y
    ix = torch.randint(len(data), (batch_size,))
    # for each batch generate 4 predictions
    x = torch.stack([data[i][j : j + block_size] for i in ix for j in range(4)])
    y = torch.stack([data[i][j + 1: j + block_size + 1] for i in ix for j in range(4)]) #
    if device != None:
      x, y = (x.to(device), y.to(device))
    return x, y

xb, yb = get_batch("train") # shape (batch_size, block_size), (batch_size), xb (B, T), yb (B, T)
targets = ignore_indexes(yb, ignore_val=11, ignore_index=-1) # change targets before equal sign(11) to -1 to not compute on lo
xb[:4], yb[:4], targets[:4]

(tensor([[12, 12, 12,  3,  7, 10,  8,  9, 11],
         [12, 12,  3,  7, 10,  8,  9, 11,  6],
         [12,  3,  7, 10,  8,  9, 11,  6,  2],
         [ 3,  7, 10,  8,  9, 11,  6,  2,  1]]),
 tensor([[12, 12,  3,  7, 10,  8,  9, 11,  6],
         [12,  3,  7, 10,  8,  9, 11,  6,  2],
         [ 3,  7, 10,  8,  9, 11,  6,  2,  1],
         [ 7, 10,  8,  9, 11,  6,  2,  1, 12]]),
 tensor([[-1, -1, -1, -1, -1, -1, -1, -1,  6],
         [-1, -1, -1, -1, -1, -1, -1,  6,  2],
         [-1, -1, -1, -1, -1, -1,  6,  2,  1],
         [-1, -1, -1, -1, -1,  6,  2,  1, 12]]))

In [ ]:
def ignore_indexes(yb, ignore_val=11, ignore_index=-1):
    # yb is shape (B, T)
    # find rows and cols where yb == ignore_val
    indices = torch.where(yb == ignore_val)
    coords = list(zip(indices[0].tolist(), indices[1].tolist()))

    # clone so we don't overwrite the original
    updated_yb = yb.clone()

    for row, col in coords:
        # set all values from start up to col (inclusive) to ignore_index
        updated_yb[row, :col+1] = ignore_index

    return updated_yb

In [ ]:
# hyperparameters
batch_size = 32
batch_size = batch_size//4 # correct for each batch(equation) we generate 4 batches.
block_size = 9 # (;;;nn+nn=)
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 10
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # generate a small batch of data of inputs x and targets y
    ix = torch.randint(len(data), (batch_size,))
    # for each batch generate 4 predictions
    x = torch.stack([data[i][j : j + block_size] for i in ix for j in range(4)])
    y = torch.stack([data[i][j + 1: j + block_size + 1] for i in ix for j in range(4)]) #
    if device != None:
      x, y = (x.to(device), y.to(device))
    return x, y

xb, yb = get_batch("train") # shape (batch_size, block_size), (batch_size), xb (B, T), yb (B, T)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = ignore_indexes(targets, ignore_val=11, ignore_index=-1) # change targets before equal sign(11) to -1 to not compute on loss
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-1)

        return logits, loss


    """def generate(self, equation):
      # list of equations to solve
      solutions = []
      # for equation in equations:
      idx = torch.tensor([encode(equation)], dtype=torch.long, device=device) # idx is the context (the equation to solve)
      predicted_idx = torch.empty((1, 1), dtype=torch.long, device=device) # predicted_idx is the predicted solution
      while True:
        # crop idx to the last block_size tokens
        idx_cond = idx[:, -block_size:]
        # get the predictions
        logits, loss = self(idx_cond)
        # focus only on the last time step
        logits = logits[:, -1, :] # becomes (B, C)
        # apply softmax to get probabilities
        probs = F.softmax(logits, dim=-1) # (B, C)
        # sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
        # append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        predicted_idx = torch.cat((predicted_idx, idx_next), dim=1) # (B, T+1)
        if idx_next.item() == 12: # end when it predicts ";"
          break
      solutions.append(predicted_idx)
      return solutions"""

    def generate(self, idx):
        # idx is (B, T) array of indices in the current context
        equations = idx
        solutions = []
        for equation in equations:
          context = equation.view(1, -1).clone() # create context shape 1, block_size
          while True:
              # crop idx to the last block_size tokens
              idx_cond = context[:, -block_size:]
              # get the predictions
              logits, loss = self(idx_cond)
              # focus only on the last time step
              logits = logits[:, -1, :] # becomes (B, C)
              # apply softmax to get probabilities
              probs = F.softmax(logits, dim=-1) # (B, C)
              # sample from the distribution
              idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
              # append sampled index to the running sequence
              context = torch.cat((context, idx_next), dim=1) # (B, T+1)
              if idx_next.item() == 12:
                  break
          solutions.append(context.tolist()[0])
        return solutions

model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


0.005263 M parameters
step 0: train loss 2.5749, val loss 2.5832
step 100: train loss 2.1395, val loss 2.1457
step 200: train loss 1.8784, val loss 1.8833
step 300: train loss 1.7280, val loss 1.7441
step 400: train loss 1.6800, val loss 1.7002
step 500: train loss 1.6478, val loss 1.6572
step 600: train loss 1.6438, val loss 1.6543
step 700: train loss 1.6259, val loss 1.6394
step 800: train loss 1.6327, val loss 1.6301
step 900: train loss 1.6190, val loss 1.6279
step 1000: train loss 1.6190, val loss 1.6308
step 1100: train loss 1.6192, val loss 1.6263
step 1200: train loss 1.6233, val loss 1.6399
step 1300: train loss 1.6184, val loss 1.6300
step 1400: train loss 1.6212, val loss 1.6451
step 1500: train loss 1.6108, val loss 1.6248
step 1600: train loss 1.6130, val loss 1.6220
step 1700: train loss 1.6168, val loss 1.6285
step 1800: train loss 1.6158, val loss 1.6227
step 1900: train loss 1.6190, val loss 1.6294
step 2000: train loss 1.6156, val loss 1.6246
step 2100: train loss 1.

In [ ]:
equations = create_equations(20, 0, 99, with_solutions=False)
encoded_equations = torch.tensor([encode(equation) for equation in equations])
solutions = m.generate(encoded_equations)
# solution = solution[-2::-1] + [solution[-1]]
for equation, solution in zip(equations, solutions):
    print(f"equation {decode_equation(solution)[3:-1]}")

equation 76+90=138
equation 10+39=046
equation 71+48=108
equation 82+42=167
equation 16+85=088
equation 89+94=176
equation 87+67=135
equation 11+82=127
equation 85+54=188
equation 65+46=075
equation 02+46=096
equation 39+23=107
equation 27+43=091
equation 98+62=158
equation 24+28=009
equation 17+19=089
equation 09+37=061
equation 12+64=023
equation 98+69=145
equation 94+67=185
